# Neural Architectures and Representation Learning

## Week 1: From Linear Models to Representation Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/01/Week_01_From_Linear_to_Representation_Learning.ipynb)

Deep learning has revolutionized how we build models that learn from data. This week we bridge the gap between classical linear models—where the decision boundary is a hyperplane in the feature space—and *representation learning*, where the network learns to transform raw inputs into task-relevant internal representations before making a prediction.

Understanding why this transition matters is essential: it explains why neural networks can solve problems that linear models fundamentally cannot, and why the geometry of hidden-layer activations is central to both interpretability and generalization.

### Concept Map (Week 1)

- **Linear models** → Decision boundary is a **hyperplane** ($f(x) = w^\top x + b = 0$). They succeed when the data are **linearly separable** (one hyperplane can separate the classes).
- **Perceptron** → Simplest neural unit; implements one linear decision boundary (one hyperplane). Converges only when data are linearly separable; fails on non-linear structure (e.g. XOR).
- **Hidden layers** → Add non-linear transformations between input and output. The first layer learns a **feature map**: inputs are mapped to a new space of learned features.
- **Representation learning** → The network learns a feature map so that the *representation* of the data (hidden activations) is easier to classify—often (approximately) linearly separable—so the final layer can use a linear decision.
- **Universal approximation** → MLPs with one hidden layer and sufficiently many units can approximate arbitrary continuous functions. This guarantees that the architecture is expressive enough; depth and width then become a question of efficiency and learning, not expressiveness.

*Flow:* Linear models (hyperplane, linear separability) → limitation (XOR) → **hidden layers** (feature map) → **representation learning** → **universal approximation** (theoretical ceiling).

---
## 1. Theory: From Linear Boundaries to Learned Representations

### Linear decision boundaries

In a linear model (e.g. logistic regression or a linear SVM), the decision function has the form

$$f(x) = w^\top x + b$$

The set of points where $f(x) = 0$ is a **hyperplane**: a flat, oriented surface that divides the input space into two half-spaces. Classification reduces to checking which half-space a point lies in—equivalently, the sign of $f(x)$.

**Intuition.** Think of the hyperplane as a wall: everything on one side gets one label, everything on the other gets the other. The weight vector $w$ controls the orientation of this wall; the bias $b$ controls how far from the origin it sits. Learning is the process of finding the orientation and position that best separates the two classes.

This framework works well when the classes are *linearly separable*—i.e. when a single hyperplane can correctly partition all the data. The same geometry applies in arbitrarily high dimensions; we simply lose the ability to visualize it directly.

#### Pause & Reflect

- In 2D we draw a line; in 3D a plane. What does "the other side of the hyperplane" mean in 100 dimensions, and why is that still a well-defined notion?
- If you had two very similar feature vectors from different classes, what would that imply about the decision boundary?
- Why might "one hyperplane" be a strength in terms of interpretability, and a limitation in terms of flexibility?

---
## Environment

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/01/Week_01_From_Linear_to_Representation_Learning.ipynb
```

### Colab

1. Open the notebook in Colab via the badge link above.
2. Runtime → Run all. Standard Colab runtimes include the packages used in this notebook; no extra setup needed.

### 🔬 Interactive Exploration: Geometry of Decision Boundaries

> **Tool:** [TensorFlow Playground](https://playground.tensorflow.org/) — an in-browser neural network simulator that lets you train small networks visually, in real time, with no code.

#### Why we pause here

We have just seen that a linear model draws exactly one hyperplane. Before we formalize deeper architectures, it is worth *seeing* what that constraint looks like in practice—and what changes the moment we add a hidden layer. TensorFlow Playground lets you watch decision boundaries form as the network trains, making the geometry of linear separability and learned feature maps directly observable rather than purely abstract.

Spend 10–15 minutes working through the experiments below. The goal is not to optimize a model; it is to build the geometric intuition that the rest of this lecture formalises.

---

#### Guided experiments

**Experiment 1 — Linear model on a linearly separable dataset**

1. Open [playground.tensorflow.org](https://playground.tensorflow.org/).
2. In the top-left, select the **Circle** or **Gaussian** dataset (two cleanly separated blobs).
3. Remove all hidden layers (set *Hidden Layers* to 0). Features: use only $X_1$ and $X_2$.
4. Hit **Play** and let it train for ~200 steps.

*What to notice:* A single straight line cleanly separates the two classes. Loss drops quickly to near zero. This is the linear model you just read about—one hyperplane, one decision.

> **Connection:** This is $f(x) = w^\top x + b$. The Playground's orange-blue background *is* the two half-spaces defined by the hyperplane.

---

**Experiment 2 — Linear model on the XOR dataset**

1. Switch the dataset to **XOR** (the four-quadrant checkerboard, top row in Playground).
2. Keep hidden layers at 0, same features $X_1$, $X_2$.
3. Hit **Play**.

*What to notice:* The boundary is still a straight line, but it cannot separate the four quadrants. Loss plateaus high. No matter how long you train, the boundary cannot twist into the right shape.

> **Connection:** This is exactly the XOR failure discussed in the next section. The network is trying to draw one wall through a checkerboard—structurally impossible.

---

**Experiment 3 — Adding one hidden layer**

1. Keep the **XOR** dataset.
2. Add **1 hidden layer** with **2 neurons**. Keep ReLU activation.
3. Hit **Play**.

*What to notice:* The boundary is no longer a straight line. With just two hidden units, the network can sometimes separate XOR—though it may struggle. Increase to **3–4 neurons** and watch it solve the problem cleanly.

> **Connection:** The hidden layer has learned a *feature map*: it transforms the original $(X_1, X_2)$ space into a new 2D (or 4D) space in which the classes *are* linearly separable. The output layer then draws one hyperplane in that learned space.

---

**Experiment 4 — Increasing hidden layer width**

1. Still on XOR, one hidden layer. Try **2, 4, 8** neurons.
2. Observe how the decision boundary changes in complexity and how fast the loss drops.

*What to notice:* More neurons → richer feature map → more flexible boundary. With 8 neurons the network can overfit to any noise. There is a tradeoff between expressive power and generalisation.

> **Connection:** This previews the Universal Approximation Theorem: more units = more expressiveness. But width alone is not the whole story.

---

**Experiment 5 — Changing the activation function**

1. In the top bar, switch *Activation* from **ReLU** to **Tanh**, then to **Linear**.
2. Try each on the XOR dataset with 1 hidden layer, 4 neurons.

*What to notice:* With **Linear** activation, the network collapses back to a single hyperplane regardless of how many layers you add—stacking affine maps is still just one affine map. With **Tanh** or **ReLU**, the boundary bends. The non-linearity is what gives the hidden layer its power.

> **Connection:** This is why removing the activation function destroys the expressive capacity of a hidden layer—a point formalised in the next section.

---

#### Pause & Reflect

Take a few minutes to discuss the following with a partner or jot down your thoughts before reading on.

1. **Boundary shape:** In Experiment 3, the decision boundary curves and bends even though the output layer is still linear. Where does this curvature come from? What is the hidden layer actually doing to the input space?

2. **Failure mode:** In Experiment 2, the loss stops decreasing even after hundreds of steps. Is the network "stuck" in a local minimum, or is the model class itself incapable of solving the problem? How would you tell the difference?

3. **Width vs. complexity:** In Experiment 4, adding more neurons makes the boundary more complex. Is a more complex boundary always better? At what point might it hurt you?

4. **The role of activation:** In Experiment 5, switching to a **Linear** activation collapses the network. From what you observed: why does stacking multiple linear layers not help? Can you articulate this in one precise sentence?

5. **Connecting to MNIST:** The network we will train later in this notebook has 784 inputs, 128 hidden units, and 10 outputs—all in a space you cannot visualise. Based on what you saw in Playground, what do you expect the hidden layer to be *doing* to the 784-dimensional pixel vectors?

### Perceptron intuition

The **perceptron** is the simplest instantiation of this idea: a single unit that takes inputs $x$, computes $z = w^\top x + b$, and produces a binary label via $\operatorname{sign}(z)$. Geometrically, it implements exactly one linear decision boundary.

**Intuition.** The perceptron is literally "one neuron": it sums up weighted evidence, adds a bias offset, and fires a yes/no response. The scalar $z$ carries both the prediction and a measure of confidence—its magnitude tells you how far the point is from the boundary, and its sign tells you which side it is on. One perceptron is therefore equivalent to one linear classifier: one wall, one decision.

The training procedure is correspondingly direct: learning adjusts $w$ and $b$ iteratively, correcting each mistake by nudging the boundary toward the misclassified point. The perceptron algorithm is guaranteed to converge in finitely many steps when the data are linearly separable; if they are not, it has no convergence guarantee and may cycle indefinitely.

#### Pause & Reflect

- How is the perceptron's update rule (moving the boundary when it makes a mistake) similar to or different from gradient descent on a loss?
- If you stacked two perceptrons in a row (output of first → input of second), would you still have a single linear boundary? Why or why not?
- Why does "linearly separable" matter for convergence? What goes wrong when the data are not linearly separable?

### Why linear models fail: the XOR example

To see where linearity breaks down, consider **XOR** (exclusive-or)—the standard demonstration that linear models have a fundamental structural limitation. Given two binary inputs $(x_1, x_2)$, the label is 1 when exactly one input is 1, and 0 otherwise. The four points $(0,0)$, $(1,1)$ are class 0, and $(0,1)$, $(1,0)$ are class 1.

The layout has a striking consequence: class 0 occupies two diagonally opposite corners of the unit square, and so does class 1. No hyperplane—regardless of orientation—can correctly partition all four points. Any linear classifier must fail.

**Intuition.** Imagine plotting the four points on paper and trying to draw a single straight line that puts both zeros on one side and both ones on the other. No matter how you tilt the line, you always trap at least one point on the wrong side. The problem structure is inherently non-linear: you need at least two line segments, or a curved boundary, which lies entirely outside the expressive power of a single linear model.

This is not an isolated edge case but a structural limitation of the entire linear model family: **a single hyperplane can only partition the space in one way**. Any problem whose decision boundary requires more than one linear cut—“diagonal” or “checkerboard” patterns—cannot be solved without first transforming the inputs into a more suitable representation.

#### Pause & Reflect

- Can you think of a simple transformation of $(x_1, x_2)$ after which XOR *would* become linearly separable? (Hint: add a third coordinate.)
- In real data (e.g. images), what might "checkerboard-like" or "XOR-like" structure correspond to?
- If you had three classes arranged in a triangle in 2D, would one linear classifier (one line) suffice? What about two lines?

### Hidden layers as learned feature maps

The solution to the XOR problem—and to non-linearity in general—is to add *hidden layers*. A **multilayer perceptron (MLP)** inserts one or more such layers between the input and the output, each applying an affine transformation followed by a non-linear *activation function* (e.g. ReLU).

The key insight is that each hidden layer produces a new *representation* of the input. Every hidden unit computes a weighted sum of its inputs and then applies a non-linearity; the resulting activation vector is a transformed view of the data. Subsequent layers operate in this transformed space rather than on the raw input.

This chain of transformations has a profound consequence: instead of hand-designing task-specific features, the network **learns a feature map** end-to-end—one that makes the final classification problem as easy as possible for the last layer. This is the core idea behind *representation learning*.

**Intuition.** Think of the hidden layer as a committee of specialists: each neuron has learned to respond to a particular pattern in the input (an edge, a curve, a co-activation of certain pixels). Their combined activations form a compact, task-relevant “summary” of the input. The output layer then needs only to separate these summaries—often a far simpler problem than separating raw pixels. In essence, the network simultaneously learns *what to look for* and *how to decide*, in a single end-to-end process.

#### Pause & Reflect

- How does the idea of "learned feature detectors" relate to the XOR problem? What could a hidden layer compute that would make XOR linearly separable?
- Why is the non-linearity (e.g. ReLU) essential? What would happen if we removed it and kept only affine layers?
- In the MNIST MLP we use later, the hidden layer has 128 units. What might "too few" vs "enough" hidden units mean for the geometry of the representation?

### Universal Approximation Theorem (intuitive view)

How expressive are MLPs in principle? The **Universal Approximation Theorem** provides a theoretical answer: a feedforward network with a *single* hidden layer, a finite but sufficiently large number of neurons, and a non-linear activation function (e.g. sigmoid or ReLU) can approximate any continuous function on a compact domain to any desired accuracy.

The geometric picture is clear: with enough hidden units, the network can place arbitrarily many “bumps” or ridge functions in the input space and superpose them to approximate even the most complex input–output mapping. Deeper networks exploit compositionality, often achieving equivalent representational power with far fewer total parameters.

**Intuition.** Think of each hidden unit as contributing one “hump” or “step” to the overall function. With many such units, you can approximate complicated, wiggly boundaries—much like fitting a curve with many small piecewise-linear segments. The theorem guarantees that no continuous target is, in principle, out of reach. Crucially, it does *not* say how many units you need, how to train them, or how well the result will generalize; it simply establishes that the architecture is not the expressive bottleneck.

Importantly, this is an existence theorem, not a constructive one: it establishes that sufficiently wide MLPs are expressive enough for any continuous task, but leaves the questions of architecture selection, optimization, and generalization to be addressed separately—empirically and theoretically.

#### Pause & Reflect

- The theorem assumes a *single* hidden layer with *finitely many* neurons. Why might depth (more layers) still be preferred in practice?
- "Approximate to any accuracy" requires more neurons as the target function gets more complex. What might "complex" mean for a classification boundary in high dimensions?
- If MLPs can approximate any continuous function, why do we still use convolutional or recurrent architectures for images and sequences?

### Further reading

- **[Distill](https://distill.pub/)** — Clear, visual articles on ML and deep learning.
- **[CS231n: CNNs for Visual Recognition](https://cs231n.github.io/)** — Stanford course notes on neural networks and representation learning.
- **3Blue1Brown — Neural networks** — [YouTube: Neural networks (playlist)](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) — Intuitive visual explanations of how neural networks learn.

---
## 2. Practical: MNIST with a simple MLP

We train a small MLP on MNIST (handwritten digits) to see representation learning in action.

The model has one hidden layer with 128 units and ReLU activation; the output layer has 10 units (one per digit) with logits for cross-entropy loss.

### Imports and device

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# Optional CUDA; fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

### Define MLP (one hidden layer, 128 units, ReLU)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x, return_hidden=False):
        x = x.view(x.size(0), -1)
        h = torch.relu(self.fc1(x))
        logits = self.fc2(h)
        if return_hidden:
            return logits, h
        return logits

model = MLP(input_size=784, hidden_size=128, num_classes=10).to(device)
print(model)

### Training loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, train loss: {avg_loss:.4f}")

### Plot training loss

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_losses, "o-")
plt.xlabel("Epoch")
plt.ylabel("Train loss")
plt.title("Training loss (MNIST MLP)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Representation Learning Demo

We use the trained MLP's hidden layer as a *representation* of each input. By passing a test batch through the model with `return_hidden=True`, we obtain the 128-dimensional hidden activations.

We then project them to 2D with PCA and plot a scatter, colored by digit label, to visualize how well the network has separated the classes in representation space.

In [ ]:
# Get one test batch and forward with hidden activations
model.eval()
data, labels = next(iter(test_loader))
data = data.to(device)
with torch.no_grad():
    logits, hidden = model(data, return_hidden=True)

# Move to CPU and numpy for PCA and plotting
H = hidden.cpu().numpy()
y = labels.numpy()

In [ ]:
# PCA to 2D
pca = PCA(n_components=2)
Z = pca.fit_transform(H)
print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")

In [ ]:
# Colored scatter by digit
plt.figure(figsize=(8, 6))
scatter = plt.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=15, alpha=0.7)
plt.colorbar(scatter, label="Digit")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Hidden representations (PCA 2D) — colored by digit")
plt.tight_layout()
plt.show()

---
## 4. Reflection

### Discussion questions (representation geometry)

1. **Cluster structure**: In the PCA plot of hidden representations, some digits form tighter clusters than others. What might explain this? How could the geometry of the data (e.g. digit variability, confusion pairs like 4/9) influence the shape of these clusters?

2. **Linear separability after the hidden layer**: The last layer of the MLP is linear. Effectively, the network learns a map from pixel space to a 128-D space in which a *linear* classifier (the last layer) can separate the 10 digits. How does this relate to the XOR argument? What does it mean for the “effective” decision boundaries in the original 784-D pixel space?

3. **PCA vs. learned representation**: We used PCA to visualize the 128-D hidden codes in 2D. PCA finds directions of maximal variance and ignores class labels. How might the 2D PCA view differ from the “true” geometry that the linear classifier uses in 128-D? When might PCA hide or distort the separation that the network actually exploits?

### TODO exercises

- **Change hidden size**: Re-define the MLP with a different `hidden_size` (e.g. 64 or 256), re-run training, and compare training loss curves and the PCA plot of hidden representations. How does representation dimensionality affect both optimization and the geometry you see?

- **Change activation**: Replace ReLU in the hidden layer with another activation (e.g. `torch.tanh` or `torch.sigmoid`), keep the same architecture and training setup, and train again. Compare loss and representation plots. What differences do you notice, and why might the choice of activation matter for representation learning?